
# C7-cnn-transfer — Session 1: Convolution and Feature Maps

*One class session, roughly 85 minutes. Prerequisites: C6-pytorch
(torch tensors, the float64 course convention, `nn.Module`,
`nn.Parameter(..., requires_grad=False)`, manual-weight modules) and,
through it, the NumPy craft of F1: slicing, broadcasting, axis
aggregation, seeded randomness, `plt.imshow`.*

**This session:** every network so far treated its input as a flat
vector of features.
Images break that habit: a pixel means little on its own and a lot
next to its neighbors, and the layer built for that locality is the
**convolution**.
This session builds it from scratch — a sliding local weighted sum,
first in 1-D by hand, then in 2-D, in NumPy component form before any
torch — then meets torch's `nn.Conv2d`, stacks kernels into **feature
maps**, works out how the **receptive field** grows when convolutions
compose, and closes with the **feature hierarchy**: what early layers
versus late layers of a deep convolutional network respond to.
Everything is hand-set weights and forward passes, the C6 register;
nothing here trains.


In [ ]:

import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn

torch.set_default_dtype(torch.float64)   # course convention (C6 Session 1)
SEED = 20260804

print("torch", torch.__version__)



## 1. Convolution in 1-D: a Sliding Local Weighted Sum

**Motivation.**
C5/C6's dense layer connects *every* input to every output: one row of
weights per output unit, each row as long as the whole input.
For a signal — a sequence of measurements, a row of pixels — that is
both wasteful and wrong-headed: the pattern "a sudden jump" looks the
same at position 3 as at position 300, so the detector for it should
be a *small* set of weights *reused at every position*.
That is the whole idea of convolution:

> Slide a short weight vector (the **kernel**) along the input; at
> each position, take the weighted sum of the entries under it.

For an input $x$ of length $n$ and a kernel $k$ of length $K$, the
output entry at position $i$ is

$$y_i \;=\; \sum_{j=0}^{K-1} x_{i+j}\, k_j ,
\qquad i = 0, 1, \dots, n - K .$$

Two conventions to pin immediately:

- **No flip.** Classical signal processing reverses the kernel before
  sliding; deep learning — and torch's `nn.Conv1d`/`nn.Conv2d` — does
  **not** (the operation is technically *cross-correlation*, but the
  field says "convolution" and so does this course).
  The formula above, with $k$ read left to right, is the one everything
  in this unit uses.
- **Valid mode.** The kernel only sits where it fully fits, so the
  output is *shorter* than the input: length $n - K + 1$.
  (Padding, which restores length, arrives in Section 5.)

**Worked by hand.** Take $x = (2, 1, 0, -1, 3, 1)$ and
$k = (1, 2, -1)$; here $n = 6$, $K = 3$, so the output has
$6 - 3 + 1 = 4$ entries:

| window | weighted sum | $y_i$ |
|---|---|---|
| $(2, 1, 0)$ | $2\cdot1 + 1\cdot2 + 0\cdot(-1)$ | $4$ |
| $(1, 0, -1)$ | $1 + 0 + 1$ | $2$ |
| $(0, -1, 3)$ | $0 - 2 - 3$ | $-5$ |
| $(-1, 3, 1)$ | $-1 + 6 - 1$ | $4$ |

So $y = (4, 2, -5, 4)$.
Every output number is a *local* verdict: the same three weights,
asked at four different positions.

### Checkpoint 1

1. For $x = (1, 0, 2, -1)$ and $k = (3, 1)$, compute the full valid
   output by hand (length first, then the entries).
2. In the worked example, which single output entry changes if
   $x_5$ (the final $1$) becomes $10$ — and why do the others survive
   untouched?
3. A length-100 input meets a length-9 kernel in valid mode: output
   length? And with a length-1 kernel — what does the operation reduce
   to?



## 2. Component Form in NumPy, and a First Detector

The hand table is three lines of NumPy: one slice, one product, one
sum, repeated at each position.
This **component form** is the reference implementation the rest of
the unit reconciles against.


In [ ]:

def conv1d_valid(x, k):
    '''Valid-mode 1-D convolution (no flip): y[i] = sum_j x[i+j] * k[j].'''
    x = np.asarray(x, dtype=np.float64)
    k = np.asarray(k, dtype=np.float64)
    n_out = x.shape[0] - k.shape[0] + 1
    y = np.zeros(n_out)
    for i in range(n_out):
        y[i] = (x[i:i + k.shape[0]] * k).sum()
    return y


x = np.array([2.0, 1.0, 0.0, -1.0, 3.0, 1.0])
k = np.array([1.0, 2.0, -1.0])
print("y =", conv1d_valid(x, k))



`y = [ 4.  2. -5.  4.]` — the hand table, confirmed.

Now choose the weights *for a purpose*, exactly as C6 chose dense-layer
weights to implement a spec.
The kernel $k = (-1, 1)$ computes $y_i = x_{i+1} - x_i$: a
**jump detector**.
On a flat stretch it outputs $0$; where the signal steps up it outputs
the step height.


In [ ]:

step = np.array([0.0, 0.0, 0.0, 1.0, 1.0, 1.0, 1.0])
edges = conv1d_valid(step, np.array([-1.0, 1.0]))
print("signal:", step)
print("edges :", edges)
print("jump detected at window index:", int(np.argmax(np.abs(edges))))



`edges` is `[0. 0. 1. 0. 0. 0.]`: zero everywhere except window
index 2 — the window $(x_2, x_3) = (0, 1)$ that straddles the jump.
One tiny kernel, reused everywhere, answers "where does the signal
change?" — this is the template for every detector in this session:
**the kernel is the pattern; the output is a map of where the pattern
occurs.**

### Checkpoint 2

1. What does the kernel $k = (1, 1, 1)$ compute, up to a constant?
   What about $k = (\tfrac13, \tfrac13, \tfrac13)$?
2. Design a length-2 kernel that outputs $0$ on any flat stretch and
   responds to a step **down** with a *positive* number.
3. Run `conv1d_valid` on `step` with your kernel from 2 and check both
   claims.



## 3. Convolution in 2-D

Images make the kernel a small *patch* of weights.
A $K_h \times K_w$ kernel slides over an $H \times W$ image; at each
position it covers a patch, multiplies entrywise, and sums:

$$y_{i,j} \;=\; \sum_{a=0}^{K_h-1} \sum_{b=0}^{K_w-1}
x_{i+a,\, j+b}\; k_{a,b} .$$

Valid mode shrinks both directions:
the output is $(H - K_h + 1) \times (W - K_w + 1)$.

**Worked by hand** — a $3\times3$ kernel on a $4\times4$ image gives a
$2\times2$ output.
Compute the top-left entry $y_{0,0}$ of

$$x = \begin{pmatrix} 1&0&2&1\\ 0&1&0&0\\ 2&0&1&2\\ 1&1&0&1 \end{pmatrix},
\qquad
k = \begin{pmatrix} 1&0&0\\ 0&1&0\\ 0&0&1 \end{pmatrix}$$

The kernel picks out the main diagonal of the $3\times3$ patch:
$y_{0,0} = x_{0,0} + x_{1,1} + x_{2,2} = 1 + 1 + 1 = 3$.
The rest by code:


In [ ]:

def conv2d_valid(image, kernel):
    '''Valid-mode 2-D convolution (no flip), component form.'''
    image = np.asarray(image, dtype=np.float64)
    kernel = np.asarray(kernel, dtype=np.float64)
    kh, kw = kernel.shape
    H, W = image.shape
    out = np.zeros((H - kh + 1, W - kw + 1))
    for i in range(out.shape[0]):
        for j in range(out.shape[1]):
            out[i, j] = (image[i:i + kh, j:j + kw] * kernel).sum()
    return out


x_img = np.array([[1.0, 0.0, 2.0, 1.0],
                  [0.0, 1.0, 0.0, 0.0],
                  [2.0, 0.0, 1.0, 2.0],
                  [1.0, 1.0, 0.0, 1.0]])
k_diag = np.eye(3)
print(conv2d_valid(x_img, k_diag))



The output is `[[3. 2.], [0. 3.]]`.
The hand value $3$ sits at $y_{0,0}$; the other three entries are the
same diagonal-sum question asked at the three other window positions:

- $y_{0,1}$: patch = rows 0–2, columns 1–3; diagonal
  $x_{0,1} + x_{1,2} + x_{2,3} = 0 + 0 + 2 = 2$.
- $y_{1,0}$: patch = rows 1–3, columns 0–2; diagonal
  $x_{1,0} + x_{2,1} + x_{3,2} = 0 + 0 + 0 = 0$.
- $y_{1,1}$: checkpoint exercise.

When a hand value and a printed value ever disagree, do not argue with
the component form — print the patch itself and look:


In [ ]:

patch = x_img[0:3, 1:4]                 # the y[0, 1] window
print("patch:\n", patch)
print("diagonal:", np.diag(patch), " sum:", np.diag(patch).sum())
print("conv output y[0, 1]:", conv2d_valid(x_img, k_diag)[0, 1])



### Checkpoint 3

1. A $5\times7$ image, a $2\times3$ kernel: output shape in valid
   mode?
2. Compute $y_{1,1}$ of the worked example by hand (rows 1–3,
   columns 1–3 patch, diagonal sum) and check it against the printout.
3. What kernel makes `conv2d_valid` return (a copy of) the image with
   one row and one column shaved off — i.e., acts as the identity on
   every full patch?



## 4. Kernels as Detectors: Edges on a Synthetic Image

A dense layer's weights *are* the program (C6's slogan); a kernel's
nine numbers are a program too, and the classic programs are edge
detectors.
Build a seeded synthetic test image — a bright vertical bar, a bright
horizontal bar, mild noise — and three classic kernels:

- `K_v` — **vertical-edge detector**: each row is $(1, 0, -1)$, so
  every output is (left column) $-$ (right column) of its patch:
  large where brightness changes *horizontally*, i.e. on the flanks of
  a **vertical** structure.
- `K_h = K_v.T` — the same detector rotated: responds to
  **horizontal** structure.
- `K_blur` — all ones over 9: the patch *average*, which smooths
  instead of detecting.


In [ ]:

rng = np.random.default_rng(SEED)
img = np.zeros((12, 12))
img[:, 3] = 1.0                                # vertical bright bar at column 3
img[8, :] = 1.0                                # horizontal bright bar at row 8
img += 0.05 * rng.standard_normal((12, 12))    # mild seeded noise

K_v = np.array([[1.0, 0.0, -1.0],
                [1.0, 0.0, -1.0],
                [1.0, 0.0, -1.0]])
K_h = K_v.T.copy()
K_blur = np.ones((3, 3)) / 9.0

resp_v = conv2d_valid(img, K_v)
resp_h = conv2d_valid(img, K_h)
resp_b = conv2d_valid(img, K_blur)

fig, axes = plt.subplots(1, 4, figsize=(11, 3))
for ax, m, title in zip(axes, [img, resp_v, resp_h, resp_b],
                        ["image", "K_v response", "K_h response", "K_blur response"]):
    ax.imshow(m, cmap="gray")
    ax.set_title(title)
    ax.axis("off")
plt.tight_layout()
plt.show()

print("K_v strongest |response| per output column:",
      np.abs(resp_v).max(axis=0).round(2))
print("K_h strongest |response| per output row:   ",
      np.abs(resp_h).max(axis=1).round(2))



Read the two printed profiles against the geometry:

- `K_v`'s column profile peaks at output columns **1 and 3**
  (≈ `2.97` and `3.11`) and is near-noise everywhere else.
  The bar lives at input column 3; an output column $j$ compares input
  columns $j$ and $j+2$, so columns $1$ and $3$ are exactly the two
  windows with the bar on one flank — opposite flanks, hence opposite
  signs in the raw map.
- `K_h`'s row profile peaks at output rows **6 and 8** for the same
  reason (bar at input row 8).
- The blur response has no peaks at all — averaging is not detecting.

Two morals.
First, **detector orientation**: `K_v` finds vertical things by
measuring *horizontal* change; its transpose does the opposite —
swapping the roles is the classic first bug.
Second, a detector reports *where* its pattern is: the response is a
**map** of the image, not a single verdict.
That map is about to get a name.

### Checkpoint 4

1. Predict, then check: which output *rows* does `K_v` light up on —
   does the horizontal bar at row 8 excite the vertical-edge detector
   anywhere?
2. Why two peak columns (1 and 3) rather than one at the bar itself?
   What sign does each carry, and what single change to `K_v` swaps
   the signs?
3. The noise term is $0.05\,\mathcal{N}(0,1)$ per pixel.
   Roughly how large can a pure-noise `K_v` response get (sum of six
   noise pixels with $\pm1$ weights), and is the printed off-peak
   level consistent with that?



## 5. `nn.Conv2d`: the Same Sum, at Torch Scale

Torch packages the sliding sum as a module, and — like C6's
`DenseLayer` — its weights are numbers you can set by hand.
The shape conventions to memorize:

- **Inputs are 4-D**: `(N, C, H, W)` — batch, **channels**, height,
  width. A grayscale image is `(1, 1, H, W)`; RGB is `(1, 3, H, W)`.
- **Weights are 4-D**: `(out_channels, in_channels, K_h, K_w)` — one
  $C_{\text{in}} \times K_h \times K_w$ *stack* of kernels per output
  channel.
- With `in_channels > 1`, each output channel slides its kernel stack
  over **all** input channels and **sums across channels**: one number
  per position, however many channels came in.

Two constructor knobs change the output grid:

- `stride=s`: move the window $s$ pixels at a time — the output grid
  is coarser.
- `padding=p`: surround the image with $p$ rings of zeros before
  sliding — the window may now sit partly "outside".

**The output-size formula** (memorize; per dimension):

$$n_{\text{out}} \;=\;
\left\lfloor \frac{n_{\text{in}} + 2p - K}{s} \right\rfloor + 1 .$$

Sanity checks: valid mode is $p = 0, s = 1$, giving $n - K + 1$;
"same" padding for odd $K$ at $s = 1$ is $p = (K-1)/2$, giving $n$
exactly.

Rebuild Section 4's `K_v` as an `nn.Conv2d` and reconcile against the
component form — C6's zero-gap discipline:


In [ ]:

conv_v = nn.Conv2d(in_channels=1, out_channels=1, kernel_size=3, bias=False)
conv_v.weight = nn.Parameter(torch.as_tensor(K_v).reshape(1, 1, 3, 3),
                             requires_grad=False)   # C6 register: hand-set and fixed

img_t = torch.as_tensor(img).reshape(1, 1, 12, 12)  # (N, C, H, W)
resp_torch = conv_v(img_t)

print("input ", tuple(img_t.shape), " -> output", tuple(resp_torch.shape))
gap = float(np.abs(resp_torch.squeeze().numpy() - resp_v).max())
print("gap vs conv2d_valid:", gap, " within 1e-12:", gap < 1e-12)



Output `(1, 1, 10, 10)` and `gap` ≈ `4.4e-16` — *not* exactly zero,
and that is worth a pause.
Both sides run the same float64 arithmetic, but torch sums the nine
products in a different **order** than the component form, and float
addition is not associative (F1's lesson), so the results may differ
in the last bit.
The course register for same-dtype reconciliations of this kind:
demand `gap < 1e-12` — far below anything meaningful, far above the
last bit.
(`bias=False` because a detector needs no offset; with a bias, one
number per output channel is added to every position of that map.)

And the formula, exercised on the knobs:


In [ ]:

def out_size(n, k, s=1, p=0):
    return (n + 2 * p - k) // s + 1

for n, k, s, p, label in [(12, 3, 1, 0, "valid 12, K=3"),
                          (12, 3, 1, 1, "same  12, K=3, p=1"),
                          (12, 3, 2, 1, "stride 2, p=1"),
                          (224, 7, 2, 3, "resnet stem: 224, K=7, s=2, p=3")]:
    print(f"{label:28s} -> {out_size(n, k, s, p)}")

probe = nn.Conv2d(1, 1, kernel_size=3, stride=2, padding=1, bias=False)
print("torch agrees:", tuple(probe(torch.zeros(1, 1, 12, 12)).shape))



`12, 12, 6, 112` — and torch's stride-2 output is `(1, 1, 6, 6)`.
Keep the last line of the table in mind: a $7\times7$ kernel at
stride 2 with padding 3 turns $224$ into $112$.
Session 2 meets that exact layer as the first thing inside ResNet-50.

### Checkpoint 5

1. Without running: `nn.Conv2d(3, 8, kernel_size=5, padding=2)` on an
   input of shape `(2, 3, 64, 64)` — output shape, and the weight
   tensor's shape?
2. For a $3\times3$ kernel, what padding gives "same" size at
   stride 1? Why does the formula's floor never bite there for even
   $n$?
3. An RGB input and `out_channels=1`: how many numbers does one output
   position sum over (kernel $3\times3$, no bias), and why is the
   answer *not* $9$?



## 6. Feature Maps: a Bank of Detectors, Stacked

Section 4 ran three detectors in three separate calls.
Real convolutional layers run a whole **bank** at once: with
`out_channels = F`, the layer holds $F$ kernel stacks and produces $F$
2-D response maps, stacked along the channel dimension.
Each of those maps is a **feature map**: *feature* because the kernel
detects one pattern, *map* because the response is laid out over image
positions.

$$\text{input } (N, C_{\text{in}}, H, W)
\;\longrightarrow\;
\text{output } (N, F, H', W')$$

The channel dimension changes meaning as data flows: at the input,
channels are *colors*; after one layer, channels are *pattern
responses* — channel 0 might be "vertical edges here", channel 1
"horizontal edges here".
Deep networks then treat those maps as a new multi-channel image and
convolve *again* — detectors of detectors, which is where Section 8's
hierarchy comes from.

---

**Worked exam-style example 1 (constrained coding).**

*Build `bank`, an `nn.Conv2d(1, 2, kernel_size=3, bias=False)` whose
output channel 0 applies `K_v` and channel 1 applies `K_h` (Section 4's
kernels), with weights registered via
`nn.Parameter(..., requires_grad=False)`.
Run it on the Section 4 image and produce `maps` of shape
`(1, 2, 10, 10)`.
Deliverables: `maps`; `gap_v`, `gap_h` — the maximum absolute
disagreement of channels 0 and 1 against the NumPy `conv2d_valid`
references (each must be `< 1e-12`, the Section 5 register); `argmax_h`
— the `(row, col)` of the largest entry of channel 1, as a tuple of
plain ints.
**Banned (zero points): `torch.nn.functional.conv2d`; loops and
comprehensions.***

---

*Solution.*
The one subtlety is the weight shape: `(2, 1, 3, 3)` — two kernel
stacks, each with one input channel.
`np.stack` builds it without loops:


In [ ]:

W_bank = torch.as_tensor(np.stack([K_v, K_h])).reshape(2, 1, 3, 3)

bank = nn.Conv2d(1, 2, kernel_size=3, bias=False)
bank.weight = nn.Parameter(W_bank, requires_grad=False)

maps = bank(img_t)
print("maps shape:", tuple(maps.shape))

gap_v = float(np.abs(maps[0, 0].numpy() - resp_v).max())
gap_h = float(np.abs(maps[0, 1].numpy() - resp_h).max())
print("gap_v:", gap_v, " gap_h:", gap_h)

flat_idx = int(maps[0, 1].argmax())
argmax_h = (flat_idx // maps.shape[-1], flat_idx % maps.shape[-1])
print("argmax_h:", argmax_h)



`maps` is `(1, 2, 10, 10)`, both gaps sit at the float64 last bit
(`8.9e-16` and `4.4e-16`, comfortably `< 1e-12`), and
`argmax_h = (8, 9)`.
Row 8 is the flank where the bright bar sits under `K_h`'s $+1$ row
(the other flank, row 6, carries the *negative* peak — compare
Section 4's row profile, which took absolute values); within that
flank row, the noise makes column 9 the largest entry.
The graded skills: the 4-D weight layout, the registration idiom, and
the stacked-channel reading of the output.

### Checkpoint 6

1. A layer maps `(1, 3, 32, 32)` to `(1, 24, 30, 30)`.
   How many kernel stacks does it hold, what is each stack's shape,
   and what patterns do its output channels index — colors or
   detector responses?
2. Extend the worked example to a third channel holding `K_blur`
   (state the new weight shape and the one line that changes).
   What would `gap_blur` compare against?
3. Why does `argmax` on a 2-D map need the divmod decoding step —
   what does `.argmax()` return on a multi-dimensional tensor?



## 7. The Receptive Field: What One Output Sees

One output of a single $3\times3$ convolution depends on a
$3\times3$ patch of input — its **receptive field** (RF).
Stack a second $3\times3$ conv on the first's output and each final
number depends on a $3\times3$ patch *of the intermediate map*, whose
entries each depend on a $3\times3$ input patch: the final RF is
$5\times5$.
Locality compounds.

**The growth rule.**
Track two numbers layer by layer (per dimension): the RF size $r$ and
the **jump** $J$ — the input-pixel distance between neighboring
positions of the current map (the product of all strides so far).
Starting from $r_0 = 1$, $J_0 = 1$, a layer with kernel $K$ and
stride $s$ updates

$$r \;\leftarrow\; r + (K - 1)\cdot J,
\qquad
J \;\leftarrow\; J \cdot s .$$

*Why:* the new output spans $K$ positions of the previous map; those
positions are $J$ input pixels apart, so they stretch the field by
$(K-1) \cdot J$ input pixels beyond a single position's $r$.
The stride multiplies the spacing *for the next layer*, not for this
one — apply the $r$ update **before** the $J$ update.

**Stride-1 special case:** $J$ stays $1$ and
$r_L = 1 + \sum_{\ell=1}^{L} (K_\ell - 1)$ —
each layer adds its kernel size minus one.

**Hand-computed, three stacked convs** ($3\times3$, $3\times3$,
$5\times5$, all stride 1):
$r = 1 + 2 + 2 + 4 = 9$.

**Empirical check** — feed a 1-D impulse through all-ones kernels
(so nothing cancels): the count of nonzero outputs is exactly the
number of output positions whose window reaches the impulse, i.e. the
RF:


In [ ]:

def rf(kernels, strides):
    '''Receptive-field size of a conv stack via the growth rule.'''
    r, J = 1, 1
    for K, s in zip(kernels, strides):
        r = r + (K - 1) * J
        J = J * s
    return r


def ones_conv1d(K):
    c = nn.Conv1d(1, 1, kernel_size=K, bias=False)
    c.weight = nn.Parameter(torch.ones(1, 1, K), requires_grad=False)
    return c


stack = nn.Sequential(ones_conv1d(3), ones_conv1d(3), ones_conv1d(5))
impulse = torch.zeros(1, 1, 41)
impulse[0, 0, 20] = 1.0
out = stack(impulse)

measured = int((out != 0).sum())
print("growth rule:", rf([3, 3, 5], [1, 1, 1]), " measured:", measured)



Both print `9`.
The rule is exact, and the measurement idiom (impulse in, count
nonzeros out) recurs in this unit's practice as the honest check on
any RF derivation.

---

**Worked exam-style example 2 (multiple choice).**

*A network begins with a $7\times7$ convolution at stride 2 followed
by a $3\times3$ convolution at stride 1.
The receptive field of one output of the second layer, in input
pixels, is:*

A. 9  B. 10  C. 11  D. 13  E. 21

*Solution.*
$r_0 = 1, J_0 = 1$.
Layer 1: $r = 1 + (7-1)\cdot 1 = 7$, then $J = 2$.
Layer 2: $r = 7 + (3-1)\cdot 2 = 11$.
**Answer C.**
The traps: forgetting the stride multiplies the second layer's
contribution gives $7 + 2 = 9$ (A); adding kernel sizes $7 + 3$ gives
$10$ (B); $7 + 3\cdot2 = 13$ misapplies the rule to $K$ instead of
$K-1$ (D); multiplying kernels, $7\cdot3$, gives $21$ (E).

---

### Checkpoint 7

1. Four stacked $3\times3$ stride-1 convs: RF by the stride-1 formula?
   How many such layers to reach an RF of at least $31$?
2. Redo the worked MC with the stride on the *second* layer instead
   ($7\times7$ s1, then $3\times3$ s2): what changes and why?
3. In the empirical check, why must the kernels be all-ones (what
   could a hand-set kernel with mixed signs do to the count)?



## 8. The Feature Hierarchy: Early Layers vs Late Layers

Stack many convolutional layers and two things grow with depth:
the receptive field (Section 7) and the *abstraction* of what each
channel detects.
The consensus picture — stated here as a fact about trained networks,
observed across architectures:

> **edges → textures → parts → objects.**
> Early layers respond to local, high-frequency structure (edges,
> spots, color transitions); middle layers to textures and repeated
> motifs; late layers to object parts and whole semantic categories.

Why this *must* be the direction of travel follows from what you have
already built: an early unit's receptive field is a few pixels — it
*cannot* respond to "dog", only to what fits in a tiny window (an
edge).
A late unit sees most of the image through hundreds of compositions —
it *can* integrate evidence into something semantic, and its map
varies slowly and is often silent (most locations contain no dog).

That contrast is measurable.
Two **synthetic** activation stacks — built with seeds, no network
needed — model the two regimes:

- `early_like`: dense, high-frequency maps (rectified white noise);
- `late_like`: sparse, blobby maps (noise smoothed twice by a
  $5\times5$ box average, then shifted down before rectifying —
  most entries die, and the survivors form smooth blobs).

Two statistics separate them:

- **activation fraction** — the share of entries that are nonzero
  (low = sparse, a late signature);
- **roughness** — the mean absolute neighbor-to-neighbor difference,
  scaled by the map's own mean absolute deviation
  (high = high-frequency, an early signature).


In [ ]:

def box_blur(m, K=5):
    out = np.zeros((m.shape[0] - K + 1, m.shape[1] - K + 1))
    for i in range(out.shape[0]):
        for j in range(out.shape[1]):
            out[i, j] = m[i:i + K, j:j + K].mean()
    return out


def act_frac(stack):
    return float((stack > 0).mean())


def roughness(stack):
    dx = np.abs(np.diff(stack, axis=-1)).mean()
    dy = np.abs(np.diff(stack, axis=-2)).mean()
    scale = np.abs(stack - stack.mean(axis=(-2, -1), keepdims=True)).mean()
    return float((dx + dy) / (2 * scale))


rng = np.random.default_rng(SEED)
noise = rng.standard_normal((8, 32, 32))

early_like = np.maximum(noise, 0.0)
late_like = np.maximum(
    np.stack([box_blur(box_blur(ch)) for ch in noise]) - 0.05, 0.0)

print(f"early_like: act_frac {act_frac(early_like):.3f}  "
      f"roughness {roughness(early_like):.3f}")
print(f"late_like : act_frac {act_frac(late_like):.3f}  "
      f"roughness {roughness(late_like):.3f}")

fig, axes = plt.subplots(1, 2, figsize=(6.5, 3))
axes[0].imshow(early_like[0], cmap="gray"); axes[0].set_title("early-like map")
axes[1].imshow(late_like[0], cmap="gray");  axes[1].set_title("late-like map")
for ax in axes: ax.axis("off")
plt.tight_layout(); plt.show()



The numbers land far apart: `early_like` activates about half its
entries (`0.499`) and is rough (`1.212`); `late_like` activates well
under half (`0.394`) and is roughly four times smoother (`0.312`).
The pictures say the same: static versus blobs.

One more depth signature, from shapes rather than statistics: in real
architectures the maps **shrink spatially and multiply in channels**
with depth — early layers hold a few high-resolution maps, late
layers hold *thousands* of tiny ones (Session 2 will read the exact
progression $56\times56\times256 \to 7\times7\times2048$ off
ResNet-50).
Coarse grids cannot represent fine spatial detail, which is the
shape-level echo of "late layers answer *what*, not precisely
*where*."

The full depth story, in one table:

| | receptive field | responds to | map statistics | grid |
|---|---|---|---|---|
| **early** | small | edges, local high-frequency pattern | dense, rough | large, few channels |
| **late** | large | parts, semantic categories | sparse, smooth | small, many channels |

### Checkpoint 8

1. A single feature map is $224\times224$ and another is $7\times7$,
   both from the same network on the same input.
   Which is more likely to encode "there is a wheel somewhere in the
   lower half", and which "a 45° edge at pixel (17, 60)"? Why?
2. Predict the direction of each statistic *before* rerunning:
   applying one more $5\times5$ box blur to `late_like`'s pre-rectify
   maps — does roughness go up or down? Does the RF story predict the
   same direction for deeper layers?
3. Why is `roughness` scaled by the mean absolute deviation — what
   would go wrong when comparing a map whose values live in
   $[0, 0.1]$ against one living in $[0, 10]$ without it?



## 9. Common Pitfalls I

**Pitfall 1 — expecting the flipped-kernel convolution.**
References written for signal processing flip the kernel; torch does
not.
For symmetric kernels the difference is invisible — for asymmetric
ones it silently negates/reverses the response:


In [ ]:

x_p = np.array([0.0, 0.0, 1.0, 0.0, 0.0])
k_asym = np.array([1.0, 0.0, -1.0])
ours = conv1d_valid(x_p, k_asym)
flipped = conv1d_valid(x_p, k_asym[::-1])
print("no flip (torch/deep learning):", ours)
print("flipped (signal processing)  :", flipped)
print("np.convolve('valid')         :", np.convolve(x_p, k_asym, mode="valid"))



`[-1. 0. 1.]` versus `[ 1. 0. -1.]` — and note the last line:
NumPy's `np.convolve` *is* the flipping kind, so it matches the
"flipped" row, not ours.
Reconciling a torch conv against `np.convolve` "to be safe" plants a
sign bug; reconcile against the component form instead.

**Pitfall 2 — the shrinking-map surprise.**
Chain valid-mode convolutions and each one shaves $K-1$ per
dimension; deep stacks can eat the whole image:


In [ ]:

n = 12
for layer in range(1, 7):
    n = out_size(n, 5)      # valid 5x5 each time
    print(f"after layer {layer}: {n}x{n}" if n > 0 else f"after layer {layer}: nothing left")
    if n <= 0:
        break



A $12\times12$ image survives exactly two valid $5\times5$ layers
(`8`, then `4`) before the third has no room ($4 < 5$) — the printed
sizes fall `8, 4, 0`, and torch would raise at the third layer.
Real architectures pad (`padding=(K-1)//2`) precisely so depth does
not consume the grid; when they *do* shrink the grid, it is by
deliberate stride, not by accident.

**Pitfall 3 — feeding `nn.Conv2d` a 2-D image.**
The module requires the `(N, C, H, W)` axes even when both are 1:


In [ ]:

try:
    conv_v(torch.as_tensor(img))            # (12, 12) -- missing N and C
except RuntimeError as e:
    print("RuntimeError:", str(e)[:100])

print("fixed:", tuple(conv_v(torch.as_tensor(img).reshape(1, 1, 12, 12)).shape))



The error names the expectation (3-D or 4-D input, got 2-D); the fix
is the `reshape(1, 1, H, W)` at the boundary — make the batch and
channel axes explicit rather than hoping.

**Pitfall 4 — a fresh module ignores the course dtype... or does it?**
`torch.set_default_dtype(torch.float64)` at the top of this notebook
governs *modules built afterwards* too — but only in this process, and
only because the header ran first:


In [ ]:

print("fresh Conv2d weight dtype:", nn.Conv2d(1, 1, 3).weight.dtype)
print("but a float32 tensor still refuses to mix:")
try:
    conv_v(torch.zeros(1, 1, 12, 12, dtype=torch.float32))
except RuntimeError as e:
    print("RuntimeError:", str(e)[:80])



The fresh module is `torch.float64` (the default applied), yet a
stray float32 *input* still crashes against float64 weights — dtype
discipline lives at every boundary, not just at module construction.
Session 2 meets the mirror image of this crash: there the *model*
(pretrained, float32) is the fixed artifact, the course's float64
habits must yield at its boundary, and the cast goes the other way.

### Checkpoint 9

1. A teammate's edge map has the expected shape but every response
   profile is mirrored left-right relative to yours.
   Which pitfall is in play, and what one-line test settles it?
2. How many valid $3\times3$ layers can a $9\times9$ image survive,
   and what padding makes the answer "as many as you like"?
3. Classify each as fine or crash, and why:
   `conv_v(torch.zeros(1, 1, 12, 12))`,
   `conv_v(torch.zeros(4, 1, 12, 12))`,
   `conv_v(torch.zeros(12, 12))`,
   `conv_v(torch.zeros(1, 2, 12, 12))`.



## Checkpoint Answers

<details><summary><b>Checkpoint 1</b></summary>

1. Length $4 - 2 + 1 = 3$:
   $y = (1\cdot3 + 0\cdot1,\; 0\cdot3 + 2\cdot1,\; 2\cdot3 - 1\cdot1)
   = (3, 2, 5)$.
2. Only $y_3$ — the last window $(-1, 3, 1)$ is the only one
   containing $x_5$; every output depends on exactly the $K$ inputs
   under its window, so all other entries are untouched.
   (Contrast a dense layer, where every output would change.)
3. $100 - 9 + 1 = 92$.
   With $K = 1$ the output has length $100$ and the operation is just
   entrywise scaling by $k_0$.

</details>

<details><summary><b>Checkpoint 2</b></summary>

1. $k = (1, 1, 1)$ computes $3\times$ the local mean (a sum over the
   window); $(\tfrac13, \tfrac13, \tfrac13)$ is the local mean itself
   — both smooth rather than detect.
2. $k = (1, -1)$: on a flat stretch $x_i - x_{i+1} = 0$; on a step
   down, $x_i > x_{i+1}$, so the output is positive.
3. `conv1d_valid(step, np.array([1.0, -1.0]))` gives
   `[0. 0. -1. 0. 0. 0.]` — zero on the flats and $-1$ at the *upward*
   step, so on a *downward* step (reverse the signal) it would give
   $+1$: both claims check.

</details>

<details><summary><b>Checkpoint 3</b></summary>

1. $(5 - 2 + 1) \times (7 - 3 + 1) = 4 \times 5$.
2. Patch rows 1–3, columns 1–3; diagonal
   $x_{1,1} + x_{2,2} + x_{3,3} = 1 + 1 + 1 = 3$ — matching the
   printed bottom-right entry `3.`.
3. The kernel with a single $1$ (at any fixed position, say the
   top-left) and zeros elsewhere: each output copies one pixel of its
   patch, so the map is the image shifted/cropped by the kernel span
   minus one — for a $2\times2$ kernel with the $1$ at top-left,
   exactly the image minus its last row and column.

</details>

<details><summary><b>Checkpoint 4</b></summary>

1. `K_v` responds in the flank columns (1 and 3) along their *entire
   height* — the vertical bar has a flank in every row — so every
   output row contains a strong value.
   The horizontal bar adds nothing anywhere: within any patch it
   contributes equally to the left and right column sums, and the
   difference cancels it.
   Check: `np.abs(resp_v).max(axis=1)` is roughly constant (≈ 3)
   across rows, with no extra peak near rows 6–8.
2. The detector responds where brightness *changes* horizontally —
   the two flanks of the bar (windows with the bar entering on the
   right vs leaving on the left), not the bar's interior, where the
   patch is horizontally uniform.
   The flank signs are opposite (`+` where the bright column sits
   under the $+1$s, `-` where it sits under the $-1$s); negating
   `K_v` (or swapping its $\pm1$ columns) swaps them.
3. Six weighted noise pixels, each $0.05\,\mathcal{N}(0,1)$: standard
   deviation $0.05\sqrt{6} \approx 0.12$, so magnitudes up to
   $\sim 0.3$–$0.4$ (2–3 sigma across many windows) are expected —
   consistent with the printed off-peak values (≈ `0.15`–`0.32`).

</details>

<details><summary><b>Checkpoint 5</b></summary>

1. Output `(2, 8, 64, 64)` — $p = 2$ is "same" for $K = 5$; weights
   `(8, 3, 5, 5)`.
2. $p = 1$: $n + 2 - 3 + 1 = n$.
   The numerator $n - 1$ divided by stride 1 is an integer, so the
   floor is exact — no rounding for any $n$.
3. $3 \times 3 \times 3 = 27$: the kernel stack covers all three
   input channels and the layer sums across them — one $3\times3$
   kernel *per channel*, not one in total.

</details>

<details><summary><b>Checkpoint 6</b></summary>

1. $24$ stacks, each of shape `(3, 3, 3)` (three input channels,
   $3\times3$ window; the full weight is `(24, 3, 3, 3)`).
   Its output channels index detector responses; only the *input's*
   channels were colors.
2. Weight becomes `(3, 1, 3, 3)`:
   `W_bank = torch.as_tensor(np.stack([K_v, K_h, K_blur])).reshape(3, 1, 3, 3)`
   with `nn.Conv2d(1, 3, ...)`.
   `gap_blur` compares channel 2 against `conv2d_valid(img, K_blur)`
   — Section 4's `resp_b`.
3. `.argmax()` with no `dim` flattens first and returns a single
   index into the flattened tensor; row and column must be recovered
   by dividing and taking the remainder with respect to the row
   length.

</details>

<details><summary><b>Checkpoint 7</b></summary>

1. $r = 1 + 4\cdot2 = 9$.
   Need $1 + 2L \ge 31$, so $L = 15$.
2. Layer 1: $r = 1 + 6 = 7$, $J$ stays 1; layer 2:
   $r = 7 + 2\cdot1 = 9$, and only *after* that does $J$ become 2.
   The stride now stretches nothing (there is no later layer), so the
   RF is 9, not 11 — stride position matters because $J$ only affects
   layers *after* the striding one.
3. With mixed signs, two contributions to the same output can cancel
   exactly, making a genuinely-influenced output print as $0$ and the
   count undercount the RF.
   All-ones kernels make every contribution positive, so no
   cancellation is possible.

</details>

<details><summary><b>Checkpoint 8</b></summary>

1. The $7\times7$ map for the wheel (huge receptive field per
   position, coarse localization is enough for "somewhere in the
   lower half"); the $224\times224$ map for the pixel-precise edge —
   a $7\times7$ grid cannot even address pixel (17, 60).
2. Roughness goes *down* (more smoothing = less neighbor-to-neighbor
   change).
   Yes — deeper layers have larger RFs, hence slower spatial
   variation, hence lower roughness: the synthetic knob moves the
   statistic the same direction depth does.
3. Roughness would become a measure of *amplitude* rather than
   frequency: the $[0, 10]$ map's raw neighbor differences are ~100×
   larger at the same shape.
   Dividing by the map's own mean absolute deviation makes the
   statistic scale-free, so only the *shape* of the variation counts.

</details>

<details><summary><b>Checkpoint 9</b></summary>

1. Pitfall 1: their pipeline flips the kernel (or they reconciled
   against `np.convolve`).
   Test: run both implementations on an impulse with an asymmetric
   kernel like $(1, 0, -1)$ — flip shows up as a reversed/negated
   response immediately.
2. Each layer shaves 2: $9 \to 7 \to 5 \to 3 \to 1$, so four layers
   (a fifth has no room).
   `padding=1` keeps $9\times9$ forever.
3. Fine — `(1, 1, 12, 12)` is the canonical shape.
   Fine — a batch of 4; the module maps each item independently.
   Crash — 2-D input (Pitfall 3).
   Crash — `conv_v` was built with `in_channels=1`, so a 2-channel
   input mismatches its weight shape `(1, 1, 3, 3)`.

</details>
